In [ ]:
%pip install iterative-stratification

In [ ]:
import torch
import shutil

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")

import importlib

drive = importlib.import_module("google.colab.drive")

getattr(drive, "mount")("/content/drive")

from pathlib import Path

path_archive_drive = Path("/content/drive/MyDrive/indoor_object_detection.zip")
path_archive = Path("/content/indoor_object_detection.zip")

if not path_archive.exists():
    assert path_archive_drive.exists()

    shutil.copy2(path_archive_drive, path_archive)

path_root = Path("/content/indoor_object_detection")

import zipfile

with zipfile.ZipFile(path_archive) as zf:
    zf.extractall(path_root)

path_root = path_root / "Indoor Object Detection Dataset"

annotations = [
    'annotation/annotation_s1.xml',
    'annotation/annotation_s2.xml',
    'annotation/annotation_s3.xml',
    'annotation/annotation_s4.xml',
    'annotation/annotation_s5.xml',
    'annotation/annotation_s6.xml',
]

sequences = [
    'sequence_1',
    'sequence_2',
    'sequence_3',
    'sequence_4',
    'sequence_5',
    'sequence_6',
]
sequences = list(map(Path, sequences))

annotations = [path_root / annotation for annotation in annotations]

In [ ]:
height = 720
width = 1280

split_ratios = {
    "train": 0.8,
    "val": 0.1,
    "test": 0.1
}
import os
print(os.listdir(path_root))

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET


class BBox:
    def __init__(self, label: str, center_x: float, center_y: float, norm_width: float, norm_height: float):
        self.label = label
        self.center_x = center_x
        self.center_y = center_y
        self.norm_width = norm_width
        self.norm_height = norm_height

    def __repr__(self):
        return f"{self.center_x:.6f} {self.center_y:.6f} {self.norm_width:.6f} {self.norm_height:.6f}"


def convert_xml_to_boxes(
    xml_path: str | Path,
    image_height: int,
    image_width: int,
) -> dict[str, list[BBox]]:
    xml_path = Path(xml_path)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    result: dict[str, list[BBox]] = {}

    for image_elem in root.findall("./images/image"):
        image_filename = image_elem.attrib["file"]

        boxes = []

        for box_elem in image_elem.findall("box"):
            top = float(box_elem.attrib["top"])
            left = float(box_elem.attrib["left"])
            width = float(box_elem.attrib["width"])
            height = float(box_elem.attrib["height"])

            label_elem = box_elem.find("label")
            if label_elem is None or label_elem.text is None:
                raise ValueError(
                    f"Missing label for box in image {image_filename}"
                )

            label = label_elem.text.strip()

            center_x = left + width / 2.0
            center_y = top + height / 2.0

            center_x /= image_width
            center_y /= image_height
            norm_width = width / image_width
            norm_height = height / image_height

            boxes.append(BBox(label, center_x, center_y, norm_width, norm_height))

        result[image_filename] = boxes

    return result

all_boxes = [convert_xml_to_boxes(annotation, height, width) for annotation in annotations]

In [ ]:
import json

all_labels: set[str] = set()

for boxes in all_boxes:
    for box_list in boxes.values():
        for box in box_list:
            all_labels.add(box.label)

all_labels = sorted(all_labels)

import numpy as np

label_ids = np.arange(len(all_labels)).tolist()
label_map = dict(zip(all_labels, label_ids))

image_to_classes: dict[str, list[int]] = {}

for boxes in all_boxes:
    for image_filename, box_list in boxes.items():
        class_ids = set(label_map[box.label] for box in box_list)
        image_to_classes[image_filename] = sorted(class_ids)

print(json.dumps(label_map))

In [ ]:
image_to_sequence: dict[str, Path] = {}

for sequence, boxes in zip(sequences, all_boxes):
    for image_filename, box_list in boxes.items():
        image_path = path_root / sequence / image_filename

        if not image_path.exists():
            raise FileNotFoundError(f"Image file {image_path} does not exist")

        image_to_sequence[image_filename] = sequence

        label_path = image_path.with_suffix(".txt")

        with open(label_path, "w") as f:
            for box in box_list:
                f.write(f"{label_map[box.label]} {repr(box)}\n")

print(f"Total number of images: {len(image_to_sequence)}")

In [ ]:
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


def split_detection_dataset(
    image_to_classes: dict[str, list[int]],
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    random_state: int = 42,
) -> dict[str, list[str]]:
    """
    Split a detection dataset using multilabel stratification.

    Each image is associated with a list of unique class IDs present
    in that image.

    Returns:
        train_images, val_images, test_images
    """
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    image_paths = np.array(list(image_to_classes.keys()))

    all_class_ids = sorted({
        class_id
        for class_ids in image_to_classes.values()
        for class_id in class_ids
    })

    if not all_class_ids:
        raise ValueError("No class IDs found in image_to_classes")

    class_to_col = {
        class_id: col
        for col, class_id in enumerate(all_class_ids)
    }

    y = np.zeros(
        (len(image_paths), len(all_class_ids)),
        dtype=np.uint8,
    )

    for row, image_path in enumerate(image_paths):
        for class_id in image_to_classes[image_path]:
            y[row, class_to_col[class_id]] = 1

    remaining_ratio = val_ratio + test_ratio

    first_split = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=remaining_ratio,
        random_state=random_state,
    )

    train_idx, remaining_idx = next(
        first_split.split(image_paths, y)
    )

    test_fraction_of_remaining = test_ratio / remaining_ratio

    second_split = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=test_fraction_of_remaining,
        random_state=random_state,
    )

    val_relative_idx, test_relative_idx = next(
        second_split.split(
            image_paths[remaining_idx],
            y[remaining_idx],
        )
    )

    val_idx = remaining_idx[val_relative_idx]
    test_idx = remaining_idx[test_relative_idx]

    return {
        "train": image_paths[train_idx].tolist(),
        "val": image_paths[val_idx].tolist(),
        "test": image_paths[test_idx].tolist(),
    }


splits = split_detection_dataset(
    image_to_classes,
    train_ratio=split_ratios["train"],
    val_ratio=split_ratios["val"],
    test_ratio=split_ratios["test"],
)

for split, image_list in splits.items():
    print(f"[{split}] split contains {len(image_list)} images")

    split_file = path_root / f"{split}.txt"

    with split_file.open("w") as f:
        for image_filename in image_list:
            f.write(f"{image_to_sequence[image_filename] / image_filename}\n")